# Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")

model


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10cd78c10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10cd796d0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="Year in which movie was released")
    director:str=Field(description="Director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [5]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10cd78c10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10cd796d0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'Year in which movie was released', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'n

In [7]:
#with only using model
response=model.invoke("Provide details about the movie Inception")
response

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. First off, Inception is a 2010 science fiction action film directed by Christopher Nolan. I remember it\'s one of his more complex films, involving dreams and time manipulation. The main character is Dom Cobb, played by Leonardo DiCaprio. He\'s a thief who steals information by infiltrating the subconscious of his targets. The term "inception" refers to the opposite of what he usually does—planting an idea rather than stealing it.\n\nThe plot is about Cobb trying to perform an inception on a powerful businessman named Robert Fischer. The challenge is that Fischer\'s subconscious is highly protected, so Cobb and his team have to go through multiple layers of dreams. Each layer has a different time progression, right? Like, going deeper into a dream slows down the time relative to the real world. That\'s a key point because the deeper they go, the more

In [8]:
#using structured output via pydantic
structured_response=model_with_structure.invoke("Provide details about the movie Inception")
structured_response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="Year in which movie was released")
    director:str=Field(description="Director of the movie")
    rating:float=Field(description="The movie rating out of 10")

#as you wrote : include_raw=true, it will add message of 
#the LLM too
model_with_structure_2=model.with_structured_output(Movie,include_raw=True)
response=model_with_structure_2.invoke("Please tell me about the movie inception")
response


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the movie "Inception." Let me see what I need to do here. They want details about it. The tools provided include a Movie function that requires title, year, director, and rating. So I need to gather that information.\n\nFirst, I know that "Inception" was directed by Christopher Nolan. The release year was 2010. The rating is a bit tricky, but I recall it has a high rating on IMDb, maybe around 8.8. Let me confirm that. Yes, it\'s 8.8/10. \n\nWait, the function parameters require all four fields: title, year, director, and rating. I have all of those. I should structure the tool call with these details. Make sure the JSON is correctly formatted. Let me double-check the spelling of the director\'s name and the exact year. Yep, Christopher Nolan and 2010 are correct. The rating is 8.8 as a number, not a string. Alright, that should cover it. Time to put it all together in the tool_call fo

### Nested Structure

In [11]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    """A movie with details"""
    title:str
    year:int
    director:str
    cast:list[Actor]
    genres:list[str]
    budget:float | None =Field(description="Budget of the movie in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10cd78c10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10cd796d0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDetails', 'description': 'A movie with details', 'parameters': {'properties': {'title': {'type': 'string'}, 'year': {'type': 'integer'}, 'director': {'type': 'string'}, 'cast': {'items': {'properties': {'name': {'type': 'string'}, 'role': {'type': 'string'}}, 'required': ['name', 'role'], 'type': 'object'}, 'type': 'array'}, 'genres': {'i

In [12]:
response=model_with_structure.invoke("Provide the details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, director='Christopher Nolan', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Balthazar')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

## TypedDict

TypedDict provides a simpler alternative using Python's built-in typina, ideal when you don't need runtime validation. Example : when you want a movie title, which ofcourse will be string but can contain intergers too.

In [19]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """ A movie ith details """
    title:Annotated[str,...,"The title of the movie"] # ... -> means optional , which we are keping empty
    year:Annotated[int,...,"Year in which movie was released"]
    director:Annotated[str,...,"Director of the movie"]
    rating:Annotated[str,...,"The movie rating out of 10"]

movie_with_typedDict=model.with_structured_output(MovieDict)

In [20]:
response=movie_with_typedDict.invoke("Please provide the details of the movie Superman")
response

{'director': 'Richard Donner',
 'rating': '7.2',
 'title': 'Superman',
 'year': 1978}

In [21]:
from typing_extensions import TypedDict,Annotated

class Actor(TypedDict):
    name:str
    role:str

class MovieDict(TypedDict):
    """A movie with details"""
    title:str
    year:int
    director:str
    cast:list[Actor]
    genres:list[str]
    budget:float | None =Field(description="Budget of the movie in millions USD")

movie_with_typedDict2=model.with_structured_output(MovieDict)
movie_with_typedDict2

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10cd78c10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10cd796d0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDict', 'description': 'A movie with details', 'parameters': {'type': 'object', 'properties': {'title': {'type': 'string'}, 'year': {'type': 'integer'}, 'director': {'type': 'string'}, 'cast': {'type': 'array', 'items': {'description': "dict() -> new empty dictionary\ndict(mapping) -> new dictionary initialized from a mapping object's\n   

In [22]:
response=movie_with_typedDict2.invoke("Please provide the details of the movie Superman")
response

{'budget': 55000000,
 'cast': [{'name': 'Christopher Reeve', 'role': 'Superman/Clark Kent'},
  {'name': 'Margot Kidder', 'role': 'Lois Lane'},
  {'name': 'Glenn Ford', 'role': 'Perry White'},
  {'name': 'Marlon Brando', 'role': "Superman's Father"}],
 'director': 'Richard Donner',
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Superman',
 'year': 1978}

In [23]:
# To find info about the model

model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### Data Classes

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [38]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [33]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information of the person """
    name:str=Field(description="Name of the person")
    email:str=Field(description="Email of the person")
    phone:int=Field(description="Phone number of the person")

agent=create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo #Auto selects provider strategy
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from : Sam Karan, sam@123.com, +91 9999999999"}] 
})

In [34]:
print(result)

{'messages': [HumanMessage(content='Extract contact info from : Sam Karan, sam@123.com, +91 9999999999', additional_kwargs={}, response_metadata={}, id='bc05be0c-c6c3-47c4-8cb3-065d1efb695c'), AIMessage(content='{\n  "name": "Sam Karan",\n  "email": "sam@123.com",\n  "phone": 9999999999\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e788d-6ecd-7461-bb16-3ddee78c47f2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 42, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})], 'structured_response': ContactInfo(name='Sam Karan', email='sam@123.com', phone=9999999999)}


In [35]:
print(result["structured_response"])

name='Sam Karan' email='sam@123.com' phone=9999999999


In [36]:
# same thing but using TypedDict
from typing_extensions import TypedDict,Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information of the person """
    name:str 
    email:str
    phone:str

agent=create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo #Auto selects provider strategy
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from : Sam Karan, sam@123.com, +91 9999999999"}] 
})

print(result["structured_response"])

{'name': 'Sam Karan', 'email': 'sam@123.com', 'phone': '+91 9999999999'}


In [37]:
# using Data class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information of the person"""
    name:str #name of the person
    email:str #email of the person
    phone:str #phone number of the person

agent=create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from : Sam Karan, sam@123.com, +91 9999999999"}] 
})

result["structured_response"]

ContactInfo(name='Sam Karan', email='sam@123.com', phone='+91 9999999999')